# M40 lab solution: shared retrieval over MCP

**Brief.** Shared retrieval: connect the RAG-over-MCP server to two different clients (a LangGraph agent and Claude Desktop or a local agent) and compare answers.

One server, [`rag_server.py`](rag_server.py) (`larkspur-rag`: `retrieve` and `db_info`), and two clients that know nothing about each other:

1. **LangGraph agent.** `langchain.agents.create_agent` (a LangGraph graph) with the server's tools wrapped as LangChain tools.
2. **Local agent.** A hand-written MCP client loop with no agent framework, the shape of notebook `03a_creating_an_mcp_client.ipynb` in M38. It uses `SECOND_LLM`, the same chat model by default; set it to a local model (the module's local-agent project uses Ollama through the `mcp-apps` group) to compare models as well as harnesses.

Both answer the same five golden questions. Claude Desktop is a third client the README shows how to configure by hand; this notebook does not drive it.

## Setup

In [ ]:
import asyncio
import json
import os
import sys
import time
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import StructuredTool
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()
nest_asyncio.apply()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from helpers.utils import get_experientiallabs_llm  # noqa: E402
from lab_kit import load_jsonl, normalise  # noqa: E402

golden = {g["id"]: g for g in load_jsonl(Path("../data/larkspur/golden.jsonl"))}
QUESTIONS = ["g03", "g07", "g13", "g17", "g20"]
llm = get_experientiallabs_llm()
SECOND_LLM = llm  # e.g. langchain_ollama.ChatOllama(model="qwen3:8b") with the mcp-apps group installed
SERVER = StdioServerParameters(command=sys.executable, args=["rag_server.py"], cwd=str(Path(".").resolve()))
SYSTEM = ("You are Larkspur Cloud's support assistant. Use the retrieve tool to find handbook passages, answer only "
          "from them, and name each source file in brackets. If the passages do not answer, say so.")
print("models:", llm.model_name, "|", getattr(SECOND_LLM, "model_name", getattr(SECOND_LLM, "model", "?")))

## Helpers: one MCP session, tool results as text

In [ ]:
def result_text(result) -> str:
    """Join an MCP tool result's text parts; fall back to structured content."""
    parts = [c.text for c in result.content if getattr(c, "text", None)]
    if parts:
        return "\n".join(parts)
    return json.dumps(getattr(result, "structuredContent", None))


async def with_session(fn):
    with open(os.devnull, "w") as errlog:
        async with stdio_client(SERVER, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                return await fn(session)


async def describe(session):
    tools = (await session.list_tools()).tools
    info = result_text(await session.call_tool("db_info", arguments={}))
    return [t.name for t in tools], info


print(asyncio.run(with_session(describe)))

## Client 1: a LangGraph agent

Each MCP tool becomes a `StructuredTool` whose coroutine forwards the call to the open session. The JSON schema the server publishes is passed through as `args_schema`, so the agent sees the same arguments the server declares.

In [ ]:
def as_langchain_tools(session, tools) -> list[StructuredTool]:
    def make(name):
        async def call(**kwargs) -> str:
            return result_text(await session.call_tool(name, arguments=kwargs))
        return call

    return [StructuredTool.from_function(coroutine=make(t.name), name=t.name, description=t.description or t.name,
                                         args_schema=t.inputSchema) for t in tools]


async def langgraph_answers(session) -> list[dict]:
    tools = as_langchain_tools(session, (await session.list_tools()).tools)
    agent = create_agent(llm, tools, system_prompt=SYSTEM)
    out = []
    for gid in QUESTIONS:
        start = time.perf_counter()
        result = await agent.ainvoke({"messages": [HumanMessage(golden[gid]["question"])]}, {"recursion_limit": 12})
        calls = [tc["name"] for m in result["messages"] for tc in getattr(m, "tool_calls", []) or []]
        out.append({"id": gid, "client": "LangGraph agent", "tool_calls": calls,
                    "answer": result["messages"][-1].content, "seconds": round(time.perf_counter() - start, 2)})
    return out


lg = asyncio.run(with_session(langgraph_answers))
for r in lg:
    print(r["id"], r["tool_calls"], "->", r["answer"][:200])

## Client 2: a local agent with no framework

In [ ]:
MAX_TOOL_ROUNDS = 4


async def local_answers(session) -> list[dict]:
    tools = (await session.list_tools()).tools
    bound = SECOND_LLM.bind_tools([
        {"type": "function", "function": {"name": t.name, "description": t.description, "parameters": t.inputSchema}}
        for t in tools
    ])
    out = []
    for gid in QUESTIONS:
        messages = [SystemMessage(SYSTEM), HumanMessage(golden[gid]["question"])]
        calls = []
        start = time.perf_counter()
        for _ in range(MAX_TOOL_ROUNDS):
            reply = await bound.ainvoke(messages)
            messages.append(reply)
            if not reply.tool_calls:
                break
            for call in reply.tool_calls:
                calls.append(call["name"])
                text = result_text(await session.call_tool(call["name"], arguments=call["args"]))
                messages.append(ToolMessage(content=text, tool_call_id=call["id"]))
        out.append({"id": gid, "client": "local agent", "tool_calls": calls,
                    "answer": messages[-1].content, "seconds": round(time.perf_counter() - start, 2)})
    return out


local = asyncio.run(with_session(local_answers))
for r in local:
    print(r["id"], r["tool_calls"], "->", r["answer"][:200])

## Comparing the two clients

The comparison is side by side: the same question, the same server, two harnesses. `span_in_answer` is a rough check (does a golden evidence span appear in the answer after lower-casing); a judge prompt would grade meaning, and reading the pairs below is the check that counts.

In [ ]:
import pandas as pd

rows = []
for a, b in zip(lg, local):
    g = golden[a["id"]]
    def hit(text):
        return any(normalise(s) in normalise(text) for s in g["evidence_spans"])
    rows.append({"id": a["id"], "type": g["type"],
                 "langgraph_calls": len(a["tool_calls"]), "local_calls": len(b["tool_calls"]),
                 "langgraph_span": hit(a["answer"]), "local_span": hit(b["answer"]),
                 "langgraph_s": a["seconds"], "local_s": b["seconds"]})
print("models:", llm.model_name, "|", getattr(SECOND_LLM, "model_name", getattr(SECOND_LLM, "model", "?")),
      "| run:", time.strftime("%Y-%m-%d"))
print(pd.DataFrame(rows).to_markdown(index=False))

In [ ]:
for a, b in zip(lg, local):
    print(f"{a['id']}: {golden[a['id']]['question']}")
    print(f"  LangGraph agent: {a['answer']}")
    print(f"  local agent:     {b['answer']}")
    print(f"  reference:       {golden[a['id']]['reference_answer']}\n")

## What this shows

- The server is written once. Two clients with different harnesses reach the same `retrieve` tool through the same protocol, and neither imports the retriever.
- Differences between the two columns come from the clients: how many times each calls `retrieve`, which `n` it passes, and how it writes the answer. With the same model in both, those are harness differences; with a local model in `SECOND_LLM`, they are model differences too.
- The multi-hop questions (`g17`, `g20`) are where the clients are most likely to differ, because they need two retrievals or one retrieval with a larger `n`.